# Modelo hipotético: la sala de espera de una clínica

Una clínica pequeña tiene **una sala de espera** y **un consultorio**. Estas son las reglas del día:

- Los pacientes que llegan con cita **esperan en orden de llegada**: el primero en llegar es el primero en ser llamado.
- El consultorio atiende **un paciente a la vez**.
- Si llega una **urgencia**, el médico **interrumpe** la consulta actual, atiende la urgencia y después **retoma la última consulta que dejó pausada**. Si se interrumpe varias veces, se retoma primero la **más reciente**.
- Al final del día, la clínica quiere **buscar** a un paciente en el historial y saber **quién esperó más**.

## Qué estructura resuelve cada parte
| Situación de la clínica | Estructura | Regla |
|---|---|---|
| Sala de espera | **Cola** (`Cola`) | FIFO: el primero en entrar es el primero en salir |
| Consultas interrumpidas por urgencias | **Pila** (`Pila`) | LIFO: la última consulta pausada es la primera que se retoma |
| Buscar un paciente y el que más esperó | **Búsqueda secuencial** | revisar el historial uno por uno |


### 1. La clase `Pila`

Definimos la pila como una clase con una lista interna. El último elemento de la lista es el **tope**. Sus métodos son `push` (agregar al tope), `pop` (sacar del tope), `peek` (ver el tope sin sacarlo), `is_empty` y `size`.


In [5]:
class Pila:  
    def __init__(self): 
        self.stack = [] 

    def is_empty(self): 
        return len(self.stack) == 0  

    def push(self, item):  
        self.stack.append(item)  

    def pop(self): 
        return None if self.is_empty() else self.stack.pop() 

    def peek(self):  
        return None if self.is_empty() else self.stack[-1]  

    def size(self): 
        return len(self.stack) 

    def __str__(self): 
        return "Pila (fondo -> tope): " + str([str(x) for x in self.stack]) 

p = Pila()  
p.push("A")  
p.push("B")  
p.push("C") 
print(p, "| tope:", p.peek())  
print(p.pop(), p.pop(), p.pop(), p.pop())  

Pila (fondo -> tope): ['A', 'B', 'C'] | tope: C
C B A None


### 2. La clase `Cola`

Definimos la cola como clase. Se agrega por el final con `enqueue` y se saca por el frente con `dequeue`. También tiene `peek` (ver quién sigue), `is_empty` y `size`.


In [6]:
class Cola: 
    def __init__(self):  
        self.queue = []  

    def is_empty(self): 
        return len(self.queue) == 0  

    def enqueue(self, item): 
        self.queue.append(item) 

    def dequeue(self): 
        return None if self.is_empty() else self.queue.pop(0)  

    def peek(self):  
        return None if self.is_empty() else self.queue[0]  

    def size(self):  
        return len(self.queue)  

    def __str__(self):  
        return "Cola (frente -> final): " + str([str(x) for x in self.queue])  

c = Cola()  
c.enqueue("A")  
c.enqueue("B")  
c.enqueue("C") 
print(c, "| frente:", c.peek())  
print(c.dequeue(), c.dequeue(), c.dequeue(), c.dequeue())  

Cola (frente -> final): ['A', 'B', 'C'] | frente: A
A B C None


### 3. La clase `Paciente`

Cada paciente guarda su nombre, el motivo de la consulta, el minuto en que llegó y cuánto esperó antes de ser atendido.


In [7]:
class Paciente:  
    def __init__(self, nombre, motivo, llegada): 
        self.nombre = nombre 
        self.motivo = motivo 
        self.llegada = llegada 
        self.espera = 0  

    def __str__(self):  
        return "{} ({})".format(self.nombre, self.motivo) 

### 4. La clase `Consultorio`: la pila en acción

El consultorio atiende un paciente a la vez. Tiene tres métodos: `llamar_siguiente` (saca al primero de la sala de espera), `interrumpir` (pausa la consulta actual y atiende una urgencia) y `terminar` (guarda al paciente en el historial y **retoma la última consulta pausada**). Las consultas pausadas se guardan en una **pila**.



In [8]:
class Consultorio:  
    def __init__(self):  
        self.actual = None  
        self.pausados = Pila()  # PILA con las consultas interrumpidas
        self.atendidos = []  

    def llamar_siguiente(self, sala, minuto):  
        paciente = sala.dequeue()  
        if paciente is None:  
            print("[min {:>3}] La sala de espera está vacía".format(minuto))  
            return  
        paciente.espera = minuto - paciente.llegada  # espera = minuto actual - minuto de llegada
        self.actual = paciente 
        print("[min {:>3}] Atiende a {} (esperó {} min)".format(minuto, paciente, paciente.espera))  

    def interrumpir(self, urgencia, minuto): 
        if self.actual is not None:  
            self.pausados.push(self.actual) 
            print("[min {:>3}] URGENCIA: se pausa a {}".format(minuto, self.actual))  
        urgencia.espera = 0 
        self.actual = urgencia 
        print("[min {:>3}] Atiende la urgencia de {}".format(minuto, urgencia))  

    def terminar(self, minuto):  
        if self.actual is None: 
            return  
            
        self.atendidos.append(self.actual) 
        print("[min {:>3}] Termina la consulta de {}".format(minuto, self.actual))
       
        self.actual = self.pausados.pop()  
        
        if self.actual is not None: 
            print("[min {:>3}] Retoma la consulta de {}".format(minuto, self.actual))

### 5. Simulación de un día en la clínica

Ejecutamos el día completo: llegan pacientes a la sala de espera (cola), el consultorio los llama, entran urgencias que interrumpen consultas (pila) y todo se registra en el historial. Fíjate en el minuto 52: dos consultas quedan pausadas y se retoman en orden inverso.


In [9]:
# 1. PREPARACIÓN DEL ENTORNO
sala = Cola()  
consultorio = Consultorio()  

# 2. APERTURA DE LA CLÍNICA 
sala.enqueue(Paciente("Ana", "control", 0)) 
sala.enqueue(Paciente("Beto", "fiebre", 0)) 

# 3. INICIO DE ATENCIÓN 
consultorio.llamar_siguiente(sala, 10) 

# 4. INTERRUPCIÓN CRÍTICA
# En el minuto 15 entra un caso grave (Gabriel). 
consultorio.interrumpir(Paciente("Gabriel", "urgencia", 15), 15) 

# 5. FIN DE LA EMERGENCIA Y RESCATE (LIFO en acción)
# En el minuto 25, Gabriel está estabilizado. 
consultorio.terminar(25) 

# 6. FIN DE CONSULTA NORMAL Y SIGUIENTE TURNO
# Diez minutos después (minuto 35), Ana por fin termina su revisión completa y es dada de alta.
consultorio.terminar(35) 

# El sistema saca a Beto de la cola
consultorio.llamar_siguiente(sala, 35)
consultorio.terminar(50)  #

[min  10] Atiende a Ana (control) (esperó 10 min)
[min  15] URGENCIA: se pausa a Ana (control)
[min  15] Atiende la urgencia de Gabriel (urgencia)
[min  25] Termina la consulta de Gabriel (urgencia)
[min  25] Retoma la consulta de Ana (control)
[min  35] Termina la consulta de Ana (control)
[min  35] Atiende a Beto (fiebre) (esperó 35 min)
[min  50] Termina la consulta de Beto (fiebre)


### 6. Búsqueda secuencial en el historial

Con el historial del día resolvemos dos preguntas: ¿en qué posición se atendió a cierto paciente? (`seq_search`) y ¿quién esperó más? (`find_largest`). Las dos recorren la lista de principio a fin.


In [10]:
def seq_search(datos, x):  # búsqueda secuencial: devuelve la posición de x o -1 
    for i in range(len(datos)): 
        if x == datos[i]:  
            return i  
    return -1  

def find_largest(datos):  # devuelve la POSICIÓN del mayor elemento
    largest = 0  
    for i in range(1, len(datos)): 
        if datos[largest] < datos[i]:  
            largest = i 
    return largest  

historial = consultorio.atendidos  
nombres = [p.nombre for p in historial] 
esperas = [p.espera for p in historial] 
print("Orden de atención:", nombres)  


pos = seq_search(nombres, "Ana")  
print("Ana fue la atención número", pos + 1, "y esperó", esperas[pos], "minutos")  
print("¿Está Zoe en el historial?", seq_search(nombres, "Zoe"))  
mayor = find_largest(esperas)  
print("Quien más esperó fue", nombres[mayor], "con", esperas[mayor], "minutos")

Orden de atención: ['Gabriel', 'Ana', 'Beto']
Ana fue la atención número 2 y esperó 10 minutos
¿Está Zoe en el historial? -1
Quien más esperó fue Beto con 35 minutos


## Resumen
| Parte del modelo | Estructura | Clase / función |
|---|---|---|
| Sala de espera | Cola (FIFO) | `Cola`, usada en `sala` |
| Consultas interrumpidas | Pila (LIFO) | `Pila`, usada en `Consultorio.pausados` |
| Buscar un paciente | Búsqueda secuencial | `seq_search` |
| Quién esperó más | Búsqueda del mayor | `find_largest` |
| Costo de buscar | Tiempo lineal | `seq_search_contando` |

**Reto opcional:** agrega a `Consultorio` un método `urgencias_pausadas()` que devuelva cuántas consultas hay pausadas, y cambia los minutos de la simulación para provocar tres interrupciones seguidas.